# LUTM-1: NumPy simulator and enumerator

This backend evaluates batches of programs on a fixed tape. Programs use the right-aligned blank-padding convention: `BBBBp#x`. Here `B` denotes the LUTM blank symbol.

Run this notebook with **`LUTM-1` as the working directory**.

In [ ]:
from pathlib import Path

import numpy as np

required_files = ("lutm.py", "numpy_backend.py", "utils.py")
missing = [name for name in required_files if not (Path.cwd() / name).is_file()]
if missing:
    raise RuntimeError(
        "Run this notebook from the LUTM-1 repository root; missing: "
        + ", ".join(missing)
    )

from numpy_backend import NumpyUTMSimulator, evaluate_program, find_exact_program
from programs import get_program
from utils import SimulatorConfig, TaskCases, padded_program_to_string

np.set_printoptions(linewidth=120)

## Run one program on one input

Edit the registered name, input, and budgets below. `program_width` is the size of the padded program region; it must be at least the literal program length.

In [ ]:
program_name = "bit_not"
input_bits = "10110"

program_width = 96
left_budget = 104
right_budget = 24
t_max = 20_000

program = get_program(program_name).program
config = SimulatorConfig(
    program_width=program_width,
    left_budget=left_budget,
    right_budget=right_budget,
    t_max=t_max,
)
simulator = NumpyUTMSimulator(config)

In [ ]:
encoded = simulator.encode_programs([program])
padded = padded_program_to_string(
    encoded[0],
    blank_id=simulator.table.blank_id,
    zero_id=simulator.table.zero_id,
    one_id=simulator.table.one_id,
)
result = simulator.simulate(encoded, input_bits)
output = result.output_strings()[0]
reason = result.invalid_reason_names()[0]
expected = get_program(program_name).target(input_bits)

print(f"initial tape: {padded}#{input_bits}")
print(f"output:       {output!r}")
print(f"expected:     {expected!r}")
print(f"exact:        {not bool(result.invalid[0]) and output == expected}")
print(f"halted:       {bool(result.halted[0])}")
print(f"invalid:      {bool(result.invalid[0])} ({reason})")
print(f"T:            {int(result.T[0]):,}")
print(
    f"space L/R:    {int(result.left_space_used[0]):,} / "
    f"{int(result.right_space_used[0]):,}"
)
print(f"final head:   {int(result.final_heads[0]):,}")
print(f"final state:  {simulator.table.states[int(result.final_state_ids[0])]}")

## Evaluate a program on input/target pairs

Inputs may include the empty string. Targets must be nonempty binary strings. The two lists must have equal length and inputs must be unique.

In [ ]:
task_program_name = "bit_not"
task_program = get_program(task_program_name).program
inputs = ["0", "1", "00", "01", "10", "11", "101"]
targets = ["1", "0", "11", "10", "01", "00", "010"]

task = TaskCases(inputs, targets)
evaluations = evaluate_program(simulator, task_program, task)

print(f"{'input':>8}  {'target':>8}  {'output':>8}  {'exact':>5}  {'reason':>22}  {'T':>10}")
for case in evaluations:
    print(
        f"{case.input_bits!r:>8}  {case.target!r:>8}  {case.output!r:>8}  "
        f"{str(case.exact):>5}  {case.invalid_reason:>22}  {case.T:>10,}"
    )
print(f"\nall exact: {all(case.exact for case in evaluations)}")

## Enumerate programs in growing length

The order is empty, `0`, `1`, `00`, `01`, `10`, `11`, `000`, ... . Within a fixed-width region that begins as `BBBBB`, `BBBB0`, `BBBB1`, `BBB00`, ... .

Enumeration is finite for the chosen width. With `stop_first_exact=True`, the function returns as soon as the first batch containing an exact program has been evaluated.

In [ ]:
search_inputs = ["0", "1", "01", "10"]
search_targets = ["0", "1", "01", "10"]

search_program_width = 5
search_left_budget = 7
search_right_budget = 8
search_t_max = 1_000
batch_size = 64
stop_first_exact = True
print_every_batches = 1

search_config = SimulatorConfig(
    program_width=search_program_width,
    left_budget=search_left_budget,
    right_budget=search_right_budget,
    t_max=search_t_max,
)
search_simulator = NumpyUTMSimulator(search_config)
search_task = TaskCases(search_inputs, search_targets)

In [ ]:
found = find_exact_program(
    search_simulator,
    search_task,
    batch_size=batch_size,
    stop_first_exact=stop_first_exact,
    print_every_batches=print_every_batches,
)

if found is None:
    print("No exact program exists within the selected width and budgets.")
else:
    print("\nFirst exact program")
    print(f"program:            {found.program!r}")
    print(f"padded:             {found.padded_program}")
    print(f"effective length:   {found.effective_length}")
    print(f"ordinal:            {found.program_ordinal:,}")
    print(f"evaluated programs: {found.evaluated_programs:,}")
    print(f"elapsed:            {found.elapsed_seconds:.3f} s")